In [2]:
import oracledb

oracledb.init_oracle_client(
    lib_dir=r"C:\instantclient-basic-windows.x64-23.26.1.0.0\instantclient_23_0"
)

In [3]:
import pandas as pd
import oracledb

# =========================
# 1. 엑셀 파일 읽기
# =========================
file_path = "./data/공고원본_POSTING_ID포함_정리결과_카테고리분류.xlsx"
df = pd.read_excel(file_path)

# 필요한 컬럼만 사용
df = df[["POSTING_ID", "MULTI_CATEGORIES"]].copy()

# =========================
# 2. 카테고리명 -> CATEGORY_ID 매핑
# =========================
category_map = {
    "프론트엔드": 1,
    "백엔드": 2,
    "인공지능/데이터": 3,
    "인프라/보안": 4,
    "모바일": 5,
    "기타": 6
}

# =========================
# 3. INSERT할 데이터 만들기
# =========================
insert_rows = []

for _, row in df.iterrows():
    posting_id = row["POSTING_ID"]
    multi_categories = row["MULTI_CATEGORIES"]

    # MULTI_CATEGORIES가 비어 있으면 건너뛰기
    if pd.isna(multi_categories) or str(multi_categories).strip() == "":
        continue

    # 쉼표 기준 분리
    categories = [c.strip() for c in str(multi_categories).split(",") if c.strip()]

    for category_name in categories:
        category_id = category_map.get(category_name)

        # 매핑되지 않는 값이 있으면 건너뛰거나 확인용 출력
        if category_id is None:
            print(f"[경고] 매핑되지 않은 카테고리: {category_name} / POSTING_ID: {posting_id}")
            continue

        insert_rows.append((int(posting_id), int(category_id)))

# 중복 제거
insert_rows = list(set(insert_rows))

print(f"INSERT 대상 행 수: {len(insert_rows)}")
print("예시 10개:", insert_rows[:10])

# =========================
# 4. Oracle DB 연결
# =========================
conn = oracledb.connect(
    user='campus_24KDT_LI8_p2_1',
    password='smhrd1',
    dsn="project-db-campus.smhrd.com:1524/xe"
)

cur = conn.cursor()

# =========================
# 5. INSERT 실행
# =========================
sql = """
INSERT INTO JOB_POSTING_CATEGORY (POSTING_ID, CATEGORY_ID)
VALUES (:1, :2)
"""

cur.executemany(sql, insert_rows)
conn.commit()

print("JOB_POSTING_CATEGORY 테이블에 데이터 삽입 완료")

# =========================
# 6. 연결 종료
# =========================
cur.close()
conn.close()

INSERT 대상 행 수: 1559
예시 10개: [(824, 1), (133, 4), (741, 3), (42, 2), (83, 3), (825, 2), (429, 2), (167, 2), (775, 1), (558, 3)]
JOB_POSTING_CATEGORY 테이블에 데이터 삽입 완료
